# Team MAS for a Toy Travel Agency

This notebook shows a **team-based multi-agent architecture**.
The agents collaborate toward one travel plan, but each teammate owns a clearly different section of the work.

## What Makes This a Team Instead of a Hierarchy?

In this notebook, we do **not** have a manager deciding dynamic sub-tasks.
Instead, we have a fixed modular team structure:

- one teammate chooses the destination and period,
- one chooses the flight,
- one chooses the hotel,
- one chooses the activities,
- one turns the shared board into the final itinerary.

The collaboration happens through a shared project board and clear role boundaries.

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

# One small model keeps the four tutorials easy to compare.
# The notebooks assume OPENAI_API_KEY is already set in your environment.
model = init_chat_model("openai:gpt-4.1-mini", temperature=0)

USER_REQUEST = """
Plan a 4-day spring trip from Rome.
Requirements:
- mid-range budget
- easy flights
- central hotel
- mix of food and culture
- simple daily plan
""".strip()

DESTINATIONS = {
    "Lisbon": {
        "best_period": "April to June",
        "style": "sunny, walkable, relaxed",
        "notes": "great for food, viewpoints, and compact neighborhoods",
    },
    "Barcelona": {
        "best_period": "April to June",
        "style": "lively, artistic, seaside",
        "notes": "strong mix of architecture, beach walks, and tapas",
    },
    "Prague": {
        "best_period": "April to June",
        "style": "historic, compact, lower-cost",
        "notes": "easy sightseeing with a classic old-town atmosphere",
    },
}

FLIGHTS = [
    {"destination": "Lisbon", "code": "TP-833", "price": 180, "type": "direct", "duration": "3h 05m"},
    {"destination": "Lisbon", "code": "IB-310", "price": 150, "type": "1 stop", "duration": "5h 10m"},
    {"destination": "Barcelona", "code": "VY-611", "price": 140, "type": "direct", "duration": "1h 50m"},
    {"destination": "Barcelona", "code": "IB-220", "price": 125, "type": "1 stop", "duration": "4h 00m"},
    {"destination": "Prague", "code": "FR-721", "price": 110, "type": "direct", "duration": "1h 55m"},
    {"destination": "Prague", "code": "OS-410", "price": 145, "type": "1 stop", "duration": "3h 45m"},
]

HOTELS = [
    {"destination": "Lisbon", "name": "Baixa Stay", "price_per_night": 145, "style": "central boutique hotel"},
    {"destination": "Lisbon", "name": "River Rooms", "price_per_night": 120, "style": "simple hotel near transit"},
    {"destination": "Barcelona", "name": "Born Hotel", "price_per_night": 160, "style": "central design hotel"},
    {"destination": "Barcelona", "name": "Gracia Inn", "price_per_night": 130, "style": "quiet hotel in a walkable district"},
    {"destination": "Prague", "name": "Old Town House", "price_per_night": 115, "style": "historic hotel near main sights"},
    {"destination": "Prague", "name": "City Garden Hotel", "price_per_night": 95, "style": "budget-friendly hotel with tram access"},
]

ACTIVITIES = [
    {"destination": "Lisbon", "name": "Alfama food walk", "tag": "food", "price": 35},
    {"destination": "Lisbon", "name": "Belem and river tram day", "tag": "culture", "price": 25},
    {"destination": "Barcelona", "name": "Gothic Quarter tapas evening", "tag": "food", "price": 40},
    {"destination": "Barcelona", "name": "Sagrada Familia and modernism route", "tag": "culture", "price": 32},
    {"destination": "Prague", "name": "Old Town walking tour", "tag": "culture", "price": 18},
    {"destination": "Prague", "name": "Czech dinner and jazz night", "tag": "food", "price": 30},
]


def flights_for(destination: str) -> list[dict]:
    return [item for item in FLIGHTS if item["destination"] == destination]



def hotels_for(destination: str) -> list[dict]:
    return [item for item in HOTELS if item["destination"] == destination]



def activities_for(destination: str) -> list[dict]:
    return [item for item in ACTIVITIES if item["destination"] == destination]



def catalog_text() -> str:
    lines = []
    for destination, info in DESTINATIONS.items():
        lines.append(f"Destination: {destination}")
        lines.append(f"  Best period: {info['best_period']}")
        lines.append(f"  Style: {info['style']}")
        lines.append(f"  Notes: {info['notes']}")
        lines.append("  Flights:")
        for flight in flights_for(destination):
            lines.append(
                f"    - {flight['code']} | {flight['type']} | EUR {flight['price']} | {flight['duration']}"
            )
        lines.append("  Hotels:")
        for hotel in hotels_for(destination):
            lines.append(
                f"    - {hotel['name']} | EUR {hotel['price_per_night']} per night | {hotel['style']}"
            )
        lines.append("  Activities:")
        for activity in activities_for(destination):
            lines.append(
                f"    - {activity['name']} | {activity['tag']} | EUR {activity['price']}"
            )
        lines.append("")
    return "\n".join(lines).strip()


CATALOG_TEXT = catalog_text()
print(USER_REQUEST)

## Define the Team Members

Each teammate has a non-overlapping responsibility.
This is useful in tutorials because it makes the handoff between agents extremely explicit.

Notice that every teammate reads the same project board, but each one writes only their own section.

In [ ]:
class TeamSection(BaseModel):
    content: str = Field(description="A short section that this teammate owns.")


section_llm = model.with_structured_output(TeamSection)

TEAM_ROLES = {
    "destination_designer": {
        "field": "destination_and_period",
        "prompt": "Choose the destination and travel period only.",
    },
    "flight_specialist": {
        "field": "flight",
        "prompt": "Choose the flight only.",
    },
    "hotel_specialist": {
        "field": "hotel",
        "prompt": "Choose the hotel only.",
    },
    "activity_specialist": {
        "field": "activities",
        "prompt": "Choose the activities only.",
    },
    "itinerary_writer": {
        "field": "final_writeup",
        "prompt": "Turn the finished board into a short client-ready itinerary.",
    },
}


def run_team_member(role_name: str, board: dict) -> TeamSection:
    role = TEAM_ROLES[role_name]

    # Team members have fixed, non-overlapping responsibilities.
    # They read the shared project board and update only their own section.
    return section_llm.invoke(
        [
            SystemMessage(
                content=(
                    "You are one teammate in a modular travel-agency MAS. "
                    "You have a fixed role and should only produce your own section. "
                    "Do not invent options outside the catalog.\n\n"
                    f"Your role: {role['prompt']}"
                )
            ),
            HumanMessage(
                content=(
                    f"Traveler request:\n{USER_REQUEST}\n\n"
                    f"Catalog:\n{CATALOG_TEXT}\n\n"
                    f"Shared project board so far:\n{board}\n\n"
                    f"Write the '{role['field']}' section only."
                )
            ),
        ]
    )

## Run the Team Workflow

The next cell uses a fixed order.
That is enough for a minimal team architecture because the role boundaries are already clear.

The important thing to watch is how the shared project board becomes richer after each teammate acts.

In [ ]:
project_board = {
    "destination_and_period": "",
    "flight": "",
    "hotel": "",
    "activities": "",
    "final_writeup": "",
}

# The workflow is fixed because this is a modular team.
# Each teammate owns one section and hands the shared board to the next person.
for role_name in [
    "destination_designer",
    "flight_specialist",
    "hotel_specialist",
    "activity_specialist",
    "itinerary_writer",
]:
    section = run_team_member(role_name, project_board)
    field_name = TEAM_ROLES[role_name]["field"]
    project_board[field_name] = section.content

print("=== Shared project board ===")
for key, value in project_board.items():
    print(f"{key}:\n{value}\n")

print("=== Final itinerary ===")
print(project_board["final_writeup"])

## Why This Fits the Team Architecture

- Roles are distinct and non-overlapping.
- Collaboration happens through a shared artifact.
- The flow is modular rather than manager-driven.

This makes the team structure feel different from both the flat and hierarchical versions.